In [2]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 10.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 16.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 11.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 13.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 12.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 10.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0
    Uninstalling python-dateutil-2.9.0:
      Successfully uninstalled python-dateutil-2.9.0
  Attempting uninstall: numpy
    Found existing i

In [1]:
# Standard Library
import datetime
import functools
import hashlib
import html
import io
import json
import logging
import os
import socket
import sys
import time
import unicodedata
import uuid
from datetime import datetime
from functools import lru_cache, reduce
from io import BytesIO
from typing import Any, Dict, List, Optional, Tuple, Type, Union
from urllib.parse import unquote_plus
from tabulate import tabulate
# Third-Party Libraries
import boto3
import numpy as np
import pandas as pd
import polars as pl
import polars.io as pio
import psycopg2
import awswrangler as wr
from botocore.exceptions import ClientError
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from psycopg2 import sql


In [18]:
pip install tabulate==0.9.0 qgrid==1.3.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.2/889.2 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for qgrid: filename=qgrid-1.3.1-py2.py3-none-any.whl size=1761251 sha256=29589a3bf5e06a19b3cd96c25fc48aa7f7fa6749319fc710181fe2002af8178e
  Stored in directory: /home/jovyan/.cache/pip/wheels/7f/0f/d6/33b60aa35dbdc3d3e84c44b780c85e92d8767b698843f9e256
Successfully built qgrid
Note: you may need to restart the kernel to use updated packages.


In [27]:
import ipywidgets as widgets
from IPython.display import display, HTML

In [2]:
REGION='us-east-1'

In [3]:
def setup_logger(name: str = None) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        formatter = logging.Formatter('[%(asctime)s] %(levelname)s @ line %(lineno)d: %(message)s')
        handler = logging.StreamHandler(sys.stdout)
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger

logger = setup_logger(__name__)

In [4]:
def get_secret(secret_name):

    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=REGION
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        logger.error(f"Failed to retrieve secret: {e}")
        raise
    else:
        if 'SecretString' in get_secret_value_response:
            secret = json.loads(get_secret_value_response['SecretString'])
            return secret
        else:
            raise ValueError("Secret value is not a string")

@functools.lru_cache(maxsize=1)
def get_db_connection_params() -> Dict[str, str]:
    logger.info("Getting database connection parameters...")
    required_params = ('ENDPOINT', 'PORT', 'DBNAME')
    params = {param: os.environ.get(param) for param in required_params}

    if not all(params.values()):
        missing = [k for k, v in params.items() if not v]
        error_msg = f"Missing required environment variables: {', '.join(missing)}"
        logger.error(error_msg)
        raise ValueError(error_msg)

    logger.info("Successfully retrieved database connection parameters")
    return  params

In [5]:
def get_predefined_query_df(query_type: str) -> pd.DataFrame:
    logger.info(f"Getting predefined query DataFrame for type: {query_type}")

    try:
        credentials = get_secret(os.environ.get("SECRET_NAME"))
        logger.info("Credentials retrieved successfully.")
        db_params = get_db_connection_params()

        conn = psycopg2.connect(
            host=db_params['ENDPOINT'],
            port=db_params['PORT'],
            database=db_params['DBNAME'],
            user=credentials['username'],
            password=credentials['password'],
            sslmode='require'
        )

        queries = {
            # 'ESTANDAR': f"SELECT {ESTANDAR_COLUMNS} FROM {SCHEMA_NAME}.source_support",
            # 'ESPECIALES': f"SELECT {ESPECIALES_COLUMNS} FROM {SCHEMA_NAME}.source_sps",
            # 'WB': f"SELECT {WB_COLUMNS} FROM {SCHEMA_NAME}.source_wb",
            'ISOS': f"SELECT {ISO_COLUMNS} FROM {SCHEMA_NAME}.source_isos",
            # 'DAPRESSURE':  f"SELECT {DAP_COLUMNS} FROM {SCHEMA_NAME}.source_dapressure",
            # 'WELDING_INFO':  f"SELECT {WELDING_INFO_COLUMNS} FROM {SCHEMA_NAME}.source_welding_info"

        }

        logger.info(f"Executing {query_type} query")
        df = pd.read_sql(queries[query_type], conn)
        logger.info(f"Successfully retrieved {len(df)} rows for {query_type}")

        conn.close()
        return df

    except Exception as e:
        logger.error(f"Error executing predefined query {query_type}: {str(e)}")
        if 'conn' in locals() and conn is not None:
            conn.close()
        raise

In [51]:

def get_predefined_query_pl(query_type: str, iso_columns: str) -> pl.DataFrame:
    logger.info(f"Getting predefined query DataFrame for type: {query_type}")
    ISO_COLUMNS = iso_columns
    try:
        credentials = get_secret(os.environ.get("SECRET_NAME"))
        logger.info("Credentials retrieved successfully.")
        db_params = get_db_connection_params()
        
        # Create connection string
        conn_string = (
            f"postgresql://{credentials['username']}:{credentials['password']}"
            f"@{db_params['ENDPOINT']}:{db_params['PORT']}/{db_params['DBNAME']}"
            "?sslmode=require"
        )
        
        # Create connection using psycopg2
        conn = psycopg2.connect(conn_string)
        
        SCHEMA_NAME = 'user_01'
        
        queries = {
            'ISOS': f"SELECT {ISO_COLUMNS} FROM {SCHEMA_NAME}.source_isos",
        }
        
        # Using Polars to read from PostgreSQL with connection object
        df = pl.read_database(
            query=queries[query_type],
            connection=conn
        )
        
        # Close the connection
        conn.close()
        
        logger.info(f"Successfully retrieved {df.height} rows for {query_type}")
        return df

    except Exception as e:
        logger.error(f"Error executing predefined query {query_type}: {str(e)}")
        raise


In [56]:
ISO_COLUMNS='col_1366,designarea,e3did,pi_d,comm_crono,system,subsystem_description,insulation,bom,sum_m,inst,flan,valv,cap,gask,fbli,tee,redu,elbo,pcom,bend,unio,vtwa,filt,trap'
SCHEMA_NAME='user_01'

In [57]:
pl_isos =get_predefined_query_pl('ISOS',ISO_COLUMNS)

[2025-05-18 11:19:35,305] INFO @ line 2: Getting predefined query DataFrame for type: ISOS
[2025-05-18 11:19:36,110] INFO @ line 6: Credentials retrieved successfully.
[2025-05-18 11:19:37,692] INFO @ line 34: Successfully retrieved 1361 rows for ISOS


In [59]:
print(pl_isos.columns)

['col_1366', 'designarea', 'e3did', 'pi_d', 'comm_crono', 'system', 'subsystem_description', 'insulation', 'bom', 'sum_m', 'inst', 'flan', 'valv', 'cap', 'gask', 'fbli', 'tee', 'redu', 'elbo', 'pcom', 'bend', 'unio', 'vtwa', 'filt', 'trap']


In [33]:
import ipywidgets as widgets
from IPython.display import display, HTML

def create_dataframe_viewer(df, rows_per_page=10):
    # Convert to pandas if it's a Polars DataFrame
    if hasattr(df, 'to_pandas'):
        df = df.to_pandas()
    
    total_rows = len(df)
    total_pages = -(-total_rows // rows_per_page)  # Ceiling division

    style = """
    <style>
        .dataframe-container {
            overflow-x: auto;
            max-width: 100%;
            max-height: 500px;
            border: 1px solid #ccc;
            padding: 8px;
        }
        .dataframe {
            margin: 10px 0;
            font-size: 14px;
        }
    </style>
    """

    output = widgets.Output()

    def show_page(page):
        with output:
            output.clear_output()
            start_idx = (page - 1) * rows_per_page
            end_idx = min(start_idx + rows_per_page, total_rows)
            page_df = df.iloc[start_idx:end_idx]
            html = style + f"""
            <div class="dataframe-container">
                {page_df.to_html(index=True, classes='dataframe')}
            </div>
            <p>Showing rows {start_idx + 1} to {end_idx} of {total_rows}</p>
            """
            display(HTML(html))

    page_slider = widgets.IntSlider(
        value=1,
        min=1,
        max=total_pages,
        step=1,
        description='Page:',
        continuous_update=False
    )
    page_slider.observe(lambda change: show_page(change['new']), names='value')

    show_page(1)  # Show first page on load
    display(page_slider, output)

In [60]:
create_dataframe_viewer(pl_isos, rows_per_page=15)

IntSlider(value=1, continuous_update=False, description='Page:', max=91, min=1)

Output()